# 02 · Screen the universe by a factor

> Goal: pull the top-N tickers by a factor today, then filter by another factor.

## What we'll do

1. Top 10 by momentum
2. Filter to ones that aren't already overbought (RSI < 70)
3. Visualize the relationship between two factors

In [1]:
# Setup — works with or without an API key.
# With FACTORWEAVE_API_KEY set, we use the full API (10,000+ tickers).
# Without one, we fall back to /demo/{ticker} (AAPL, MSFT, NVDA, AMZN, GOOGL, META, TSLA, JPM).
import os, json
import requests

API_BASE = "https://factorweave.com/api"
API_KEY = os.environ.get("FACTORWEAVE_API_KEY")
DEMO_TICKERS = ["AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "TSLA", "JPM"]
MODE = "live" if API_KEY else "demo"
print(f"Running in {MODE} mode.", "Key prefix:", (API_KEY[:8] + '…') if API_KEY else "(none)")


def fw_demo(ticker: str) -> dict:
    """Demo endpoint — no auth, 8 sample tickers, current snapshot only."""
    r = requests.get(f"{API_BASE}/demo/{ticker}", timeout=10)
    r.raise_for_status()
    return r.json()


def fw_features(ticker: str, **kwargs) -> dict:
    """Authed features endpoint when a key is available; demo fallback otherwise."""
    if API_KEY:
        r = requests.get(f"{API_BASE}/features/{ticker}",
                         params=kwargs,
                         headers={"X-API-Key": API_KEY},
                         timeout=10)
        r.raise_for_status()
        return r.json()
    # Demo fallback — reshape demo response to look like the authed one
    d = fw_demo(ticker)
    return {"rows": [{"ticker": d["ticker"], "date": d["as_of"], **d["factors"]}]}


def fw_top(factor: str, n: int = 25) -> dict:
    """Top-N by a factor. Needs auth for the full universe; in demo mode we
    rank the 8 demo tickers locally."""
    if API_KEY:
        r = requests.get(f"{API_BASE}/top",
                         params={"factor": factor, "n": n},
                         headers={"X-API-Key": API_KEY},
                         timeout=10)
        r.raise_for_status()
        return r.json()
    # Demo fallback — fetch each demo ticker, sort locally
    rows = []
    for t in DEMO_TICKERS:
        d = fw_demo(t)
        if factor in d["factors"]:
            rows.append({"ticker": t, "date": d["as_of"], factor: d["factors"][factor]})
    rows.sort(key=lambda r: r[factor], reverse=True)
    return {"rows": rows[:n]}


def fw_similar(ticker: str, method: str = "cosine", limit: int = 10) -> dict:
    """Similarity search. Demo endpoint includes pre-computed `similar` set."""
    if API_KEY:
        r = requests.get(f"{API_BASE}/vector-search/similar/{ticker}",
                         params={"method": method, "limit": limit, "min_lookback_days": 30},
                         headers={"X-API-Key": API_KEY},
                         timeout=10)
        r.raise_for_status()
        return r.json()
    # Demo fallback — uses the `similar` array baked into the demo response
    d = fw_demo(ticker)
    return {"ticker": ticker, "method": "cosine (demo)", "neighbors": d.get("similar", [])[:limit]}


def fw_market_context() -> dict:
    """Universe analytics. Public on FREE, fuller on HOBBY+."""
    headers = {"X-API-Key": API_KEY} if API_KEY else {}
    r = requests.get(f"{API_BASE}/market-context", params={"latest": 1}, headers=headers, timeout=10)
    if r.status_code == 401:
        return {"_note": "market-context requires auth in demo mode"}
    r.raise_for_status()
    return r.json()


Running in demo mode. Key prefix: (none)


## Top 10 momentum names today

In [2]:
import pandas as pd

top = fw_top("mom", n=10)
df = pd.DataFrame(top["rows"])
df

,ticker,date,mom
0,AAPL,2026-05-22,0.139305
1,TSLA,2026-05-22,0.132102
2,GOOGL,2026-05-22,0.111992
3,NVDA,2026-05-22,0.033898
4,AMZN,2026-05-22,0.008826
5,JPM,2026-05-22,-0.006163
6,MSFT,2026-05-22,-0.014248
7,META,2026-05-22,-0.095951


In live mode this ranks all ~10,000 tickers. In demo mode it ranks just the 8 demo tickers locally — enough to see the shape.

## Filter: strong momentum AND not overbought

To screen "strong momentum that hasn't already shot up too far", we want momentum high *and* RSI moderate.

In [3]:
import pandas as pd

# pull full row for each candidate to get RSI
candidates = []
for t in df["ticker"]:
    row = fw_features(t)["rows"][0]
    candidates.append({"ticker": t, "mom": row.get("mom"), "rsi": row.get("rsi")})

screen = pd.DataFrame(candidates)
not_overbought = screen[screen["rsi"] < 70]
print(f"{len(not_overbought)}/{len(screen)} pass the RSI<70 filter")
not_overbought

7/8 pass the RSI<70 filter


,ticker,mom,rsi
1,TSLA,0.132102,61.535813
2,GOOGL,0.111992,49.796630
3,NVDA,0.033898,62.509280
4,AMZN,0.008826,43.326345
5,JPM,-0.006163,48.795752
6,MSFT,-0.014248,54.284231
7,META,-0.095951,49.904202


## Honest framing

This is a **screening lens**, not a buy signal. Our own leak-free probes
show factor similarity (and high momentum, and any of these factors alone)
does *not* forecast forward returns. Use this output as a starting watchlist
for further research, not as an entry list.

## Next

→ `03-similarity-peer-set.ipynb` — find historical analogues of a given ticker's current factor state.